# Generating Text using Character RNN

In [1]:
import tensorflow as tf

filepath = tf.keras.utils.get_file(
    fname="shakespeare.txt", 
    origin="https://homl.info/shakespeare",
    cache_dir='datasets',
)

with open(filepath) as f:
    shakespeare_text = f.read()

print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [2]:
# Maps every character into an integer. Starting at 2. Values 0 and 1 are reserved for: padding tokens and unknown characters.
text_vec_layer = tf.keras.layers.TextVectorization(
    split="character", 
    standardize="lower"
)

text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]
encoded -= 2 # drop tokens reserved for 0 (pad) and 1 (unknown), they are not used
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded)

print("encoded ", encoded)
print("n_tokens ", n_tokens)
print("dataset_size ", dataset_size)

encoded  tf.Tensor([19  5  8 ... 20 26 10], shape=(1115394,), dtype=int64)
n_tokens  39
dataset_size  1115394


In [3]:
from scripts.character_rrn import to_dataset
SEED = 42
tf.random.set_seed(SEED)

length = 100
train_end = int(dataset_size * 0.90) # 90% for training
val_end = train_end + int(dataset_size * 0.05)

train_set = to_dataset(sequence=encoded[:train_end], length=length, shuffle=True, seed=SEED)
val_set = to_dataset(sequence=encoded[train_end:val_end], length=length)
test_set = to_dataset(sequence=encoded[val_end:], length=length)

In [4]:
train_set

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"]
)

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "shakespeare_model.keras", 
    monitor="val_accuracy",
    save_best_only=True,
)

history = model.fit(
    train_set, 
    validation_data=val_set, 
    epochs=10, 
    callbacks=[model_ckpt]
)

shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda x: x - 2),
    model
])

Epoch 1/10
  13250/Unknown 949s 69ms/step - accuracy: 0.5137 - loss: 1.6295

In [ ]:
y_proba = shakespeare_model.predict(["To be or not to b"])[0, -1]
y_pred = tf.argmax(y_proba)
text_vec_layer.get_vocabulary()[y_pred + 2]